# CALM-nanoGPT PoC — Google Colab

**CALM (Continuous Autoregressive Language Model)** を nanoGPT スタイルで実装し、
GPU を使って tiny Shakespeare データセットで学習・テキスト生成します。

## 手順
1. GPU ランタイムの確認
2. 依存ライブラリのインストール
3. リポジトリのクローン
4. データ準備
5. Autoencoder の学習
6. CALM モデルの学習
7. テキスト生成

---
**ランタイム設定**: `ランタイム → ランタイムのタイプを変更 → T4 GPU` を選択してから実行してください。

## 0. GPU 確認

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU             : {gpu.name}')
    print(f'VRAM            : {gpu.total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  GPU が検出されません。ランタイムの設定を確認してください。')
    print('   ランタイム → ランタイムのタイプを変更 → T4 GPU')

## 1. 依存ライブラリのインストール

In [ ]:
# Colab には PyTorch が既にインストール済みなので tiktoken だけ追加
!pip install tiktoken --quiet
print('✅ インストール完了')

## 2. リポジトリのクローン

> **前提**: このノートブックと同じリポジトリを GitHub に push しておく必要があります。
> 
> ローカルで:
> ```bash
> cd CALM_PoC
> git remote add origin https://github.com/YOUR_USERNAME/CALM_PoC.git
> git push -u origin main
> ```
> 
> または Google Drive を使う方法は **セル 2b** を参照。

In [ ]:
# --- 方法A: GitHub からクローン（推奨）---
# YOUR_USERNAME を自分の GitHub ユーザー名に変更
GITHUB_USER = 'YOUR_USERNAME'
REPO_NAME   = 'CALM_PoC'

import os
if not os.path.exists(REPO_NAME):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
else:
    print(f'{REPO_NAME}/ は既に存在します。git pull で最新化します...')
    !cd {REPO_NAME} && git pull

%cd {REPO_NAME}
!ls

In [ ]:
# --- 方法B: Google Drive から zip をアップロード（GitHub を使わない場合）---
# ローカルで: zip -r CALM_PoC.zip CALM_PoC/ -x '*/out-*' -x '*/__pycache__/*' -x '*.bin'
# その zip を Google Drive にアップロードし、以下を実行

# from google.colab import drive
# drive.mount('/content/drive')
# !unzip /content/drive/MyDrive/CALM_PoC.zip -d /content/
# %cd /content/CALM_PoC
# !ls

print('方法B を使う場合はこのセルのコメントを外してください。')

## 3. データ準備

tiny Shakespeare (~304K トークン) をダウンロードして学習・評価用に分割します。

In [ ]:
!python3 data/prepare_shakespeare.py

import os
for f in ['data/train.bin', 'data/val.bin']:
    size_kb = os.path.getsize(f) // 1024
    print(f'  {f}: {size_kb:,} KB')

## 4. Autoencoder の学習

4トークン → 128次元潜在ベクトル → 4トークン に変換する VAE を学習します。  
**GPU (T4) では約 3〜5 分で完了します。**

In [ ]:
!python3 train_autoencoder.py config/train_ae_colab.py

In [ ]:
# AE の再構成精度を確認
import torch, tiktoken
from autoencoder import AutoencoderConfig, Autoencoder

ckpt = torch.load('out-ae/ckpt.pt', map_location='cpu', weights_only=False)
ae = Autoencoder(AutoencoderConfig(**ckpt['ae_config']))
ae.load_state_dict(ckpt['model'])
ae.eval()

enc = tiktoken.get_encoding('gpt2')

test_sentences = [
    'To be, or not to be, that is the question:',
    'All that glitters is not gold',
    'Now is the winter of our discontent',
]

total_correct, total_tokens = 0, 0
for text in test_sentences:
    tokens = enc.encode(text)
    pad = (4 - len(tokens) % 4) % 4
    tokens_padded = tokens + [enc.eot_token] * pad
    x = torch.tensor(tokens_padded).reshape(-1, 4)
    with torch.no_grad():
        mean, _ = ae.encoder(x)
        logits = ae.decoder(mean.unsqueeze(0))
        pred = logits.argmax(-1).reshape(-1).tolist()
    orig = enc.decode(tokens_padded)
    reco = enc.decode(pred[:len(tokens_padded)])
    correct = sum(a == b for a, b in zip(tokens_padded, pred))
    total_correct += correct
    total_tokens += len(tokens_padded)
    acc = correct / len(tokens_padded) * 100
    print(f'  [{acc:.0f}%] "{text}"')
    if orig != reco:
        print(f'         → "{reco.strip()}"')

print(f'\n  総合精度: {total_correct/total_tokens*100:.1f}%')
print(f'  val_loss: {ckpt["best_val_loss"]:.4f}')

## 5. CALM モデルの学習

以下の 2 つのモードから選択できます。

| モード | config | 学習時間 (T4) | 生成品質 |
|---|---|---|---|
| **patch_size=1 (AR)** | `train_calm_colab_ar.py` | ~20分 | ✅ 意味のある文章 |
| **patch_size=4 (CALM本来)** | `train_calm_colab_k4.py` | ~60分 | 要検証 |

まず patch_size=1 で動作確認してから、patch_size=4 に挑戦することを推奨します。

In [ ]:
# ========== patch_size=1 (AR baseline) ==========
# GPU (T4) で約 20 分
!python3 train_calm.py config/train_calm_colab_ar.py

In [ ]:
# ========== patch_size=4 (true CALM) ==========
# GPU (T4) で約 60 分 — 上のセルを実行後に試してください
# !python3 train_calm.py config/train_calm_colab_k4.py

## 6. テキスト生成

In [ ]:
import torch, tiktoken
from autoencoder import AutoencoderConfig, Autoencoder
from model import CALM, CALMConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# AE の読み込み
ae_ckpt = torch.load('out-ae/ckpt.pt', map_location=device, weights_only=False)
ae = Autoencoder(AutoencoderConfig(**ae_ckpt['ae_config'])).to(device)
ae.load_state_dict(ae_ckpt['model'])
ae.eval()

# CALM モデルの読み込み（patch_size=1）
calm_ckpt = torch.load('out-calm-ar/ckpt.pt', map_location=device, weights_only=False)
calm = CALM(CALMConfig(**calm_ckpt['model_args']), ae).to(device)
calm.load_state_dict(calm_ckpt['model'], strict=False)
calm.eval()

print(f'モデル: patch_size={calm_ckpt["model_args"]["patch_size"]}, '
      f'best_val_loss={calm_ckpt["best_val_loss"]:.4f}')
print(f'デバイス: {device}')

In [ ]:
# ===== 生成パラメータ (自由に変えてみてください) =====
PROMPT             = 'ROMEO:\nO, she doth teach the torches to burn bright!'  # プロンプト
MAX_NEW_TOKENS     = 200   # 生成トークン数
TEMPERATURE        = 0.75  # 低い→保守的 / 高い→多様 (0.5〜1.2 が目安)
TOP_K              = 40    # 上位 K トークンからサンプリング (20〜100)
REPETITION_PENALTY = 1.2   # 繰り返し抑制 (1.0=無効 / 1.5=強め)
# =====================================================

enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode(PROMPT)
patch_size = calm_ckpt['model_args']['patch_size']
if patch_size > 1:
    pad = (patch_size - len(tokens) % patch_size) % patch_size
    tokens += [enc.eot_token] * pad
x = torch.tensor([tokens], dtype=torch.long, device=device)

with torch.no_grad():
    gen = calm.generate(
        x,
        max_new_patches=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        repetition_penalty=REPETITION_PENALTY,
    )

print('=' * 60)
print(enc.decode(gen[0].cpu().tolist()))
print('=' * 60)

In [ ]:
# 複数のプロンプトを一括生成
prompts = [
    'ROMEO:\nO, she doth teach the torches to burn bright!',
    'First Citizen:\nBefore we proceed any further, hear me speak.',
    'KING HENRY VI:\nMy lords, at once: the care you have of us,',
]

for prompt in prompts:
    tokens = enc.encode(prompt)
    x = torch.tensor([tokens], dtype=torch.long, device=device)
    with torch.no_grad():
        gen = calm.generate(x, max_new_patches=100, temperature=0.75, top_k=40, repetition_penalty=1.2)
    text = enc.decode(gen[0].cpu().tolist())
    print(f'\n>>> {text[:400]}')
    print()

## 7. (オプション) チェックポイントを Google Drive に保存

Colab のセッションは切断されると `/content/` が消えます。  
学習済みモデルを保存しておきたい場合は以下を実行してください。

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/CALM_PoC_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

for ckpt_dir in ['out-ae', 'out-calm-ar', 'out-calm-k4']:
    src = f'/content/CALM_PoC/{ckpt_dir}/ckpt.pt'
    dst = f'{SAVE_DIR}/{ckpt_dir}_ckpt.pt'
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / 1024**2
        print(f'  保存完了: {dst}  ({size_mb:.0f} MB)')
    else:
        print(f'  スキップ: {src} が存在しません')

## 8. (オプション) patch_size=4 の結果を比較

セル 5 で `train_calm_colab_k4.py` を実行済みの場合に使用できます。

In [ ]:
import os

if not os.path.exists('out-calm-k4/ckpt.pt'):
    print('out-calm-k4/ckpt.pt が見つかりません。セル5の patch_size=4 学習を先に実行してください。')
else:
    k4_ckpt = torch.load('out-calm-k4/ckpt.pt', map_location=device, weights_only=False)
    k4_model = CALM(CALMConfig(**k4_ckpt['model_args']), ae).to(device)
    k4_model.load_state_dict(k4_ckpt['model'], strict=False)
    k4_model.eval()

    prompt = 'ROMEO:\nO, she doth teach the torches to burn bright!'
    tokens = enc.encode(prompt)
    pad = (4 - len(tokens) % 4) % 4
    tokens_padded = tokens + [enc.eot_token] * pad
    x = torch.tensor([tokens_padded], dtype=torch.long, device=device)

    print('=== patch_size=1 (AR) val_loss={:.4f} ==='.format(calm_ckpt['best_val_loss']))
    x1 = torch.tensor([tokens], dtype=torch.long, device=device)
    with torch.no_grad():
        gen1 = calm.generate(x1, max_new_patches=80, temperature=0.75, top_k=40, repetition_penalty=1.2)
    print(enc.decode(gen1[0].cpu().tolist()[:300]))

    print('\n=== patch_size=4 (CALM) val_loss={:.4f} ==='.format(k4_ckpt['best_val_loss']))
    with torch.no_grad():
        gen4 = k4_model.generate(x, max_new_patches=20, temperature=0.75, top_k=40, repetition_penalty=1.2)
    print(enc.decode(gen4[0].cpu().tolist()[:300]))